# Debugging RAG failures: the real-library code of the lessons

Booster LEAP, Advanced RAG Engineering. The cells below are the library versions of the examples in the lessons (the lessons first build the same mechanism in plain Python in your browser). Each cell was run once and its output is saved below it.

<h1 style="color:#b45309">&#9888; First: File &rarr; Save a copy in Drive</h1>

<span style="font-size:20px">This notebook is read-only. Make your own copy now, then work in the copy (no GitHub login needed).</span>

**How to use it:** run the cells from top to bottom (Runtime, Run all, or Shift+Enter on each cell). Change one thing at a time and run again.

**What needs what:** the first run downloads the small embedding model all-MiniLM-L6-v2 (about 90 MB; it needs internet but no GPU and no key). No secret or key is used in this notebook. The install line pins the versions that were used to produce the saved outputs; Colab may resolve the dependencies of those packages differently.

In [ ]:
%pip -q install sentence-transformers==6.3.0 numpy==2.5.2 chromadb==1.5.9

## Irrelevant retrieval: find out why before you fix

One question three ways: plain words, slang, and plain words behind a wrong team filter.

In [ ]:
records = {
    "sec-devices": ("security", "A lost or stolen company laptop must be reported to the security desk within 2 hours."),
    "exp-claims": ("finance", "Submit expense claims in the finance portal within 30 days of the purchase."),
    "exp-receipts": ("finance", "Claims above 5000 rupees need a receipt and the approval of your manager."),
    "sec-password": ("security", "Passwords must have at least 14 characters and are changed only after a suspected leak."),
    "run-sev": ("engineering", "A severity 1 incident is a full outage; the on-call engineer is paged and posts an update every 30 minutes."),
    "faq-trash": ("support", "Deleted tickets stay in the trash for 30 days and are then removed for good."),
    "kb-notebook": ("support", "Notebooks in the editor can be shared with a read-only link."),
}

import chromadb
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
def embed(texts):
    return model.encode(texts, normalize_embeddings=True, show_progress_bar=False).tolist()

col = chromadb.EphemeralClient().create_collection("orbit-debug", embedding_function=None, configuration={"hnsw": {"space": "cosine"}})
col.add(ids=list(records), embeddings=embed([t for _, t in records.values()]),
        metadatas=[{"team": team} for team, _ in records.values()])

def show(label, question, where=None):
    r = col.query(query_embeddings=embed([question]), n_results=3, where=where)
    pairs = [f"{i} {1 - d:.2f}" for i, d in zip(r["ids"][0], r["distances"][0])]
    print(f"{label:13s}", " | ".join(pairs))

plain = "my work laptop was stolen, how quickly must I report it"
show("plain words", plain)
show("slang", "someone nicked my notebook, who do I tell")
show("wrong filter", plain, where={"team": "finance"})

plain words   sec-devices 0.81 | exp-claims 0.33 | exp-receipts 0.30
slang         sec-devices 0.39 | kb-notebook 0.30 | run-sev 0.21
wrong filter  exp-claims 0.33 | exp-receipts 0.30


## Duplicate chunks: when one fact fills the top k

chromadb: add and upsert with an existing id, a new id with equal text, and a repeated id in one call.

In [ ]:
import chromadb

col = chromadb.EphemeralClient().create_collection("orbit-dupes", embedding_function=None)

def state(label):
    got = col.get(include=["documents"])
    print(f"{label:26s}", col.count(), dict(sorted(zip(got["ids"], got["documents"]))))

col.add(ids=["export#0"], embeddings=[[1.0, 0.0]], documents=["link works 24 hours"])
state("first add")
col.add(ids=["export#0"], embeddings=[[0.0, 1.0]], documents=["link works 48 hours"])
state("add, same id")
col.upsert(ids=["export#0"], embeddings=[[0.0, 1.0]], documents=["link works 48 hours"])
state("upsert, same id")
col.add(ids=["run2-export-0"], embeddings=[[0.0, 1.0]], documents=["link works 48 hours"])
state("add, new id, same text")
try:
    col.add(ids=["x", "x"], embeddings=[[1.0, 0.0], [1.0, 0.0]], documents=["a", "b"])
except Exception as error:
    print("same id twice in one call:", type(error).__name__)

first add                  1 {'export#0': 'link works 24 hours'}
add, same id               1 {'export#0': 'link works 24 hours'}
upsert, same id            1 {'export#0': 'link works 48 hours'}
add, new id, same text     2 {'export#0': 'link works 48 hours', 'run2-export-0': 'link works 48 hours'}
same id twice in one call: DuplicateIDError


## Trace logging for RAG: keeping what happened

A root span with three children, read back from the in-memory exporter.

In [ ]:
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

exporter = InMemorySpanExporter()                        # keeps finished spans in a list
provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(exporter))
tracer = provider.get_tracer("orbit.help")
MS = 1_000_000                                           # span times are in nanoseconds

with tracer.start_as_current_span("rag.request", start_time=0, end_on_exit=False) as root:
    root.set_attribute("rag.index_build", "rebuild-2")
    steps = [("rag.retrieve", 0, 18, {"rag.ids": ["old-uptime", "faq-uptime"]}),
             ("rag.pack", 18, 19, {"rag.dropped": 1}),
             ("rag.generate", 19, 659, {"rag.model": "stand-in-model"})]
    for name, start, end, attributes in steps:           # times are given numbers
        child = tracer.start_span(name, start_time=start * MS, attributes=attributes)
        child.end(end_time=end * MS)
root.end(end_time=665 * MS)

spans = exporter.get_finished_spans()
top = next(s for s in spans if s.parent is None)
print("finished order:", [s.name for s in spans])
for s in sorted(spans, key=lambda s: (s.start_time, s.parent is not None)):
    depth = 0 if s.parent is None else 1
    print(f"{'  ' * depth}{s.name:{16 - 2 * depth}s} {(s.end_time - s.start_time) // MS:4d} ms  {dict(s.attributes)}")
print("one trace id for all:", len({s.context.trace_id for s in spans}) == 1)
print("children point to the root:", all(s.parent.span_id == top.context.span_id for s in spans if s.parent))

finished order: ['rag.retrieve', 'rag.pack', 'rag.generate', 'rag.request']
rag.request       665 ms  {'rag.index_build': 'rebuild-2'}
  rag.retrieve     18 ms  {'rag.ids': ('old-uptime', 'faq-uptime')}
  rag.pack          1 ms  {'rag.dropped': 1}
  rag.generate    640 ms  {'rag.model': 'stand-in-model'}
one trace id for all: True
children point to the root: True


## Drift monitoring and regression suites

Two tests on the uptime question, run with pytest on the released and the careless data.

In [ ]:
import contextlib, io, os, tempfile
from pathlib import Path
import pytest

os.environ["PYTEST_DISABLE_PLUGIN_AUTOLOAD"] = "1"
SUITE = """
DOCS = {docs}
RANKED = {ranked}          # saved search result for the uptime question

def test_current_value_in_top3():
    assert any("99.9 percent" in DOCS[i] for i in RANKED[:3])

def test_no_old_value_in_top3():
    assert not any("99.5 percent" in DOCS[i] for i in RANKED[:3])
"""
builds = {
    "good": ({'faq-uptime': 'The Business plan commits to 99.9 percent monthly uptime.', 'price-business': 'The Business plan costs 20 per seat per month and includes chat support within 4 hours.'},
             ['faq-uptime', 'price-business']),
    "bad": ({'old-uptime': 'The Business plan commits to 99.5 percent monthly uptime.', 'faq-uptime': 'The Business plan commits to 99.9 percent monthly uptime.', 'price-business': 'The Business plan costs 20 per seat per month and includes chat support within 4 hours.'},
             ['old-uptime', 'faq-uptime', 'price-business']),
}

class Collect:
    def __init__(self):
        self.results = []
    def pytest_runtest_logreport(self, report):
        if report.when == "call":
            self.results.append((report.nodeid.split("::")[-1], report.outcome))

with tempfile.TemporaryDirectory() as folder:
    for name, (docs, ranked) in builds.items():
        path = Path(folder) / f"test_{name}.py"
        path.write_text(SUITE.format(docs=docs, ranked=ranked), encoding="utf-8")
        seen = Collect()
        with contextlib.redirect_stdout(io.StringIO()):        # hide timings and paths
            code = pytest.main([str(path), "-q", "-p", "no:cacheprovider", f"--rootdir={folder}"], plugins=[seen])
        print(f"build {name}: exit code {int(code)}")
        for test, outcome in seen.results:
            print(f"   {test}: {outcome}")

build good: exit code 0
   test_current_value_in_top3: passed
   test_no_old_value_in_top3: passed
build bad: exit code 1
   test_current_value_in_top3: passed
   test_no_old_value_in_top3: failed
